## 시나리오

매주/매월 같은 포맷으로 반복해서 만드는 보고서를 자동화합니다. 이 예제는 아래 구성요소를 모두 포함합니다.

1. **표지**: 제목 + 부제 + 작성일 (큰 글씨, 가운데 정렬)
2. **요약** (Executive Summary)
3. **본문 — 데이터 표** (지역별 매출)
4. **본문 — 분석 텍스트** (하이라이트 포함)
5. **결론** (굵게 강조)
6. **HWP + PDF 두 포맷으로 저장**

## 전체 코드

```python
import datetime
from pathlib import Path
from hwpapi.core import App


def reset_formatting(app):
    """본문 기본 서식으로 복귀."""
    app.set_charshape(
        bold=False, italic=False,
        underline_type=0, strike_out_type=0,
        height=1000,             # 10pt
        text_color="#000000",
        shade_color="#FFFFFF",
    )
    app.set_parashape(align_type=1, line_spacing=160, indentation=0)


def insert_heading(app, text, level=1):
    """제목 스타일. level=1: 16pt 굵게, level=2: 13pt 굵게."""
    size = {1: 1600, 2: 1300}.get(level, 1000)
    app.insert_text(text + "\n")
    # 방금 삽입한 줄 선택
    app.api.Run("MoveLineUp")
    app.select_text()
    app.set_charshape(bold=True, height=size)
    app.move.bottom_of_file()
    reset_formatting(app)
    app.insert_text("\n")


def insert_highlight(app, text):
    """형광펜으로 강조된 문장 삽입."""
    app.set_charshape(shade_color="#FFFF00")
    app.insert_text(text)
    app.set_charshape(shade_color="#FFFFFF")


def insert_bold(app, text):
    app.set_charshape(bold=True)
    app.insert_text(text)
    app.set_charshape(bold=False)


def insert_table(app, rows, headers=None):
    n_cols = max(len(headers or []), max(len(r) for r in rows))
    n_rows = len(rows) + (1 if headers else 0)
    action = app.actions.TableCreate
    action.pset.Rows = n_rows
    action.pset.Cols = n_cols
    action.run()
    if headers:
        app.set_charshape(bold=True)
        app.set_parashape(align_type=3)
        for h in headers:
            app.insert_text(str(h))
            app.api.Run("TableRightCell")
        app.set_charshape(bold=False)
        app.set_parashape(align_type=1)
    for row in rows:
        for i in range(n_cols):
            val = row[i] if i < len(row) else ""
            app.insert_text(str(val))
            app.api.Run("TableRightCell")
    app.move.bottom_of_file()
    app.insert_text("\n")


def generate_report(
    title: str,
    subtitle: str,
    summary: str,
    sales_data: list[tuple[str, int, float]],   # (지역, 매출, 성장률)
    analysis: str,
    highlight: str,
    conclusion: str,
    output_stem: str,
):
    app = App(is_visible=True)

    # 1. 표지
    app.insert_text(title + "\n")
    app.move.top_of_file()
    app.select_text()
    app.set_charshape(bold=True, height=2400)
    app.set_parashape(align_type=3)

    app.move.bottom_of_file()
    reset_formatting(app)
    app.set_parashape(align_type=3)
    app.set_charshape(height=1400)
    app.insert_text(subtitle + "\n")

    today = datetime.datetime.now().strftime("%Y년 %m월 %d일")
    app.set_charshape(height=1100, italic=True)
    app.insert_text(f"\n{today}\n\n\n")
    reset_formatting(app)

    # 2. Executive Summary
    insert_heading(app, "1. 요약", level=1)
    app.insert_text(summary + "\n")

    # 3. 데이터 표
    insert_heading(app, "2. 지역별 매출", level=1)
    rows = [
        (region, f"{amount:,}원", f"{growth*100:+.1f}%")
        for region, amount, growth in sales_data
    ]
    insert_table(app, rows, headers=["지역", "매출", "전년비"])

    # 4. 분석 텍스트 + 하이라이트
    insert_heading(app, "3. 분석", level=1)
    app.insert_text(analysis + "\n\n")
    insert_highlight(app, highlight)
    app.insert_text("\n\n")

    # 5. 결론
    insert_heading(app, "4. 결론", level=1)
    insert_bold(app, "▸ ")
    app.insert_text(conclusion + "\n")

    # 6. HWP + PDF 저장
    stem = Path(output_stem)
    hwp_path = str(stem.with_suffix(".hwp"))
    pdf_path = str(stem.with_suffix(".pdf"))
    app.save(hwp_path)
    app.save(pdf_path)
    print(f"✅ 저장 완료:")
    print(f"  HWP: {hwp_path}")
    print(f"  PDF: {pdf_path}")
    return app


# 실행
generate_report(
    title="2026년 1분기 영업 보고",
    subtitle="― 주요 지역 매출 현황 및 성장 분석 ―",
    summary=(
        "2026년 1분기 전사 매출은 전년 동기 대비 12.5% 성장한 45.8억원을 기록했습니다. "
        "수도권 성장세가 두드러졌으며, 신규 제품 라인업 확대가 주요 요인으로 분석됩니다."
    ),
    sales_data=[
        ("서울",     1_850_000_000, 0.185),
        ("경기",     1_320_000_000, 0.142),
        ("인천",       620_000_000, 0.089),
        ("부산",       440_000_000, 0.056),
        ("기타",       350_000_000, -0.021),
    ],
    analysis=(
        "수도권 매출 비중이 전체의 83%에 달하며, 특히 서울 지역에서 "
        "신규 제품 B 라인의 성공적 런칭이 확인되었습니다. "
        "반면 '기타' 지역은 유일하게 역성장을 기록해 대책이 필요합니다."
    ),
    highlight="서울·경기 지역 성장세가 전체 실적을 견인하는 구조",
    conclusion="2분기에는 지방 거점 영업력 강화와 신제품 B의 전국 확대에 집중해야 합니다.",
    output_stem="reports/sales_2026Q1",
)
```

## 결과물 구조

```
┌─────────────────────────────────┐
│                                 │
│    2026년 1분기 영업 보고       │
│  ― 주요 지역 매출 현황 분석 ―    │
│                                 │
│        2026년 4월 15일          │
│                                 │
│  1. 요약                        │
│  2026년 1분기 전사 매출은...    │
│                                 │
│  2. 지역별 매출                 │
│  ┌─────┬──────────┬────────┐    │
│  │지역 │매출      │전년비  │    │
│  ├─────┼──────────┼────────┤    │
│  │서울 │1,850,...│+18.5%  │    │
│  │...                      │    │
│                                 │
│  3. 분석                        │
│  수도권 매출 비중이 ...         │
│  ⟦서울·경기 지역 성장세가⟧       │
│                                 │
│  4. 결론                        │
│  ▸ 2분기에는...                 │
└─────────────────────────────────┘
```

## 핵심 포인트

- **헬퍼 함수로 레이어 분리**: `insert_heading`, `insert_table`, `insert_highlight`, `insert_bold` 등으로 의도를 명확히.
- **`reset_formatting`**: 제목·강조 후 본문 서식으로 되돌리는 헬퍼를 만들어두면 서식 상태 관리가 쉬워집니다.
- **숫자 포맷팅**: 파이썬 f-string (`f"{amount:,}원"`, `f"{growth*100:+.1f}%"`) 으로 미리 포맷팅해서 표에 넣는 게 제일 깔끔합니다.
- **HWP + PDF 동시 저장**: `app.save(...)` 가 확장자로 포맷을 자동 감지하므로 `.hwp` 와 `.pdf` 둘 다 저장할 수 있습니다.

## 추가 개선

- **머리말/꼬리말**: `app.actions.HeaderFooter` + pset 으로 회사명·페이지 번호 추가
- **차트**: matplotlib 이미지로 저장 후 `InsertPicture` 액션으로 삽입
- **템플릿화**: 외부 YAML/JSON 에서 보고서 데이터를 읽어 주기적으로 자동 생성
- **메일 발송 연동**: 저장 후 `win32com` 의 Outlook 인터페이스로 자동 첨부 발송

## 📄 실행 결과 (실제 HWP 출력)

`generate_report()` 함수를 실행하면 아래 레이아웃의 HWP/PDF 보고서가 저장됩니다. 3번 분석 섹션의 주요 문장에는 형광펜 효과가 적용됩니다.

![보고서 자동 생성 결과](img/demo_report_generation.png)

::: {.callout-tip collapse="false"}
위 이미지는 **실제 HWP에서 코드를 실행 → PDF로 저장 → 렌더링**한 결과입니다.
(`tests/generate_doc_artifacts.py`)
:::

**콘솔 출력:**

```
✅ 저장 완료:
  HWP: reports/sales_2026Q1.hwp
  PDF: reports/sales_2026Q1.pdf
```
